In [1]:
import folium
from folium import plugins
import psycopg2
import json
import branca.colormap as cm


# ----------------------------
# Connect to PostGIS
# ----------------------------
conn = psycopg2.connect(
    dbname="utahdaminundationprofiles_aug9_2025",
    user="admin",
    password="admin",
    host="localhost",
    port=5432
)
cur = conn.cursor()

# ----------------------------
# Create folium map
# ----------------------------
m = folium.Map(location=[40.7607, -111.8939], zoom_start=7)

# ----------------------------
# Query all dams (including 0-intersection)
# ----------------------------
cur.execute("""

SELECT column_name
FROM information_schema.columns
WHERE table_name = 'transportation';

""")

results = cur.fetchall()


In [2]:
cur.close()
conn.close()

In [3]:
results

[('objectid',),
 ('length_mi',),
 ('length_yd',),
 ('shape_length',),
 ('geom',),
 ('pretyp',),
 ('stgeometry',),
 ('basename',),
 ('oid_1',),
 ('name',),
 ('pretypeabr',)]

In [4]:
columns = []
for item in results:
    columns.append(item[0])

In [5]:
columns

['objectid',
 'length_mi',
 'length_yd',
 'shape_length',
 'geom',
 'pretyp',
 'stgeometry',
 'basename',
 'oid_1',
 'name',
 'pretypeabr']

In [6]:
conn = psycopg2.connect(
    dbname="utahdaminundationprofiles_aug9_2025",
    user="admin",
    password="admin",
    host="localhost",
    port=5432
)
cur = conn.cursor()

# ----------------------------
# Create folium map
# ----------------------------
m = folium.Map(location=[40.7607, -111.8939], zoom_start=7)

# ----------------------------
# Query all dams (including 0-intersection)
# ----------------------------
cur.execute("""

SELECT column_name
FROM information_schema.columns
WHERE table_name = 'utah_dam_inundation_zones';

""")

results = cur.fetchall()

columns2 = []
for item in results:
    columns2.append(item[0])

cur.close()
conn.close()

In [7]:
input_prompt = {
    "prompt": "find all /transportation in 200 m from metro stations from /utah_dam_inundation_zones",
    "tables": [
        {
            "name": "transportation",
            "columns": columns
        },
        {
            "name": "utah_dam_inundation_zones",
            "columns": columns2
        }
    ]
}


In [8]:
input_prompt

{'prompt': 'find all /transportation in 200 m from metro stations from /utah_dam_inundation_zones',
 'tables': [{'name': 'transportation',
   'columns': ['objectid',
    'length_mi',
    'length_yd',
    'shape_length',
    'geom',
    'pretyp',
    'stgeometry',
    'basename',
    'oid_1',
    'name',
    'pretypeabr']},
  {'name': 'utah_dam_inundation_zones',
   'columns': ['objectid',
    'shape_length',
    'shape_area',
    'geom',
    'type',
    'dam_name',
    'nid_id',
    'federal_id',
    'primary_owner_type',
    'primary_purpose',
    'river_or_stream_name',
    'id',
    'damnumber',
    'name']}]}

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Define the model ID
model_id = "defog/sqlcoder-7b-2"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [10]:
schema = """

-- Name: transportation; Type: TABLE; Schema: public; Owner: admin
--

CREATE TABLE public.transportation (
    objectid integer NOT NULL,
    basename character varying(45),
    oid_1 character varying(14),
    pretyp smallint,
    pretypeabr character varying(10),
    name character varying(49),
    stgeometry double precision,
    length_mi double precision,
    length_yd double precision,
    shape_length double precision,
    geom public.geometry(MultiLineString,4326)
);

-- Name: utah_dam_inundation_zones; Type: TABLE; Schema: public; Owner: admin
--

CREATE TABLE public.utah_dam_inundation_zones (
    objectid integer NOT NULL,
    id character varying(1),
    damnumber character varying(7),
    name character varying(41),
    type character varying(9),
    dam_name character varying(8000),
    nid_id character varying(8000),
    federal_id character varying(8000),
    primary_owner_type character varying(8000),
    primary_purpose character varying(8000),
    river_or_stream_name character varying(8000),
    shape_length double precision,
    shape_area double precision,
    geom public.geometry(MultiPolygon,4326)
);

"""

In [11]:
user_question = "Calculate the total length of transportation where they are intersected with utah_dam_inundation_zones using PostGIS functions: ST_Intersection and ST_Length. Use EPSG code 26912 if length function need reproject. Transform to EPSG code 26912 if need."

prompt2 = f"""
### Task
Generate a SQL query to answer [QUESTION]{user_question}[/QUESTION]

### Database Schema
The query will run on a database with the following schema:
{schema}

### Answer
Given the database schema, here is the SQL query that [QUESTION]{user_question}[/QUESTION]
[SQL]
"""

In [12]:
import time
start_time = time.time()

inputs = tokenizer(prompt2, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0]))

end_time = time.time()

duration = end_time - start_time

print(f"Operation time: {duration:.4f} seconds")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> 
### Task
Generate a SQL query to answer [QUESTION]Calculate the total length of transportation where they are intersected with utah_dam_inundation_zones using PostGIS functions: ST_Intersection and ST_Length. Use EPSG code 26912 if length function need reproject. Transform to EPSG code 26912 if need.[/QUESTION]

### Database Schema
The query will run on a database with the following schema:


-- Name: transportation; Type: TABLE; Schema: public; Owner: admin
--

CREATE TABLE public.transportation (
    objectid integer NOT NULL,
    basename character varying(45),
    oid_1 character varying(14),
    pretyp smallint,
    pretypeabr character varying(10),
    name character varying(49),
    stgeometry double precision,
    length_mi double precision,
    length_yd double precision,
    shape_length double precision,
    geom public.geometry(MultiLineString,4326)
);

-- Name: utah_dam_inundation_zones; Type: TABLE; Schema: public; Owner: admin
--

CREATE TABLE public.utah_dam_inunda